# Baseline Modeling — Credit Card Fraud Detection

This notebook establishes diagnostic baseline classifiers on the preprocessed dataset.

**Objectives:**
- Train a diverse set of model families under controlled baseline settings
- Rank models using PR-AUC as the primary metric, with ROC-AUC as a secondary signal
- Shortlist the top-performing models for hyperparameter tuning

No threshold adjustment or hyperparameter search is performed at this stage.

In [1]:
import os
import pickle
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score, average_precision_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

### Reproducibility & Project Configuration

Centralized directories for model artifacts and evaluation outputs are defined, and a fixed random seed is used to ensure deterministic behavior across runs.

In [2]:
RANDOM_STATE = 42

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
PROCESSED_PATH = os.path.join(DATA_DIR, "processed")

MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: C:\Users\mail2\OneDrive\Desktop\UNSW\git_projects\fraud_detection


### 3. Load Preprocessed Data

The datasets generated in the preprocessing notebook are loaded.

- X_train: scaled and SMOTE-balanced
- X_val: scaled, original class distribution
- y_train / y_val: binary target

The validation set is used for all baseline model comparisons in this notebook. 
The test set is intentionally not loaded here — it is reserved exclusively for 
final, one-time reporting in Notebook 5, after model and threshold selection 
are already complete.

In [3]:
X_train = pd.read_csv(os.path.join(PROCESSED_PATH, "X_train.csv"))
X_val   = pd.read_csv(os.path.join(PROCESSED_PATH, "X_val.csv"))
y_train = pd.read_csv(os.path.join(PROCESSED_PATH, "y_train.csv")).values.ravel()
y_val   = pd.read_csv(os.path.join(PROCESSED_PATH, "y_val.csv")).values.ravel()

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)

Train shape: (341176, 30)
Validation shape: (56962, 30)


### Baseline Evaluation Helper

A lightweight evaluation function focused on ranking capability: PR-AUC as the primary metric, ROC-AUC as a secondary one. No threshold manipulation or plotting is performed at this stage.

In [4]:
def evaluate_baseline(model, X_val, y_val):
    probs = model.predict_proba(X_val)[:, 1]
    
    return {
        "roc_auc": roc_auc_score(y_val, probs),
        "pr_auc": average_precision_score(y_val, probs)
    }

### Baseline Model Definitions

Three model families are evaluated to assess performance under severe class imbalance:
- **Logistic Regression** — linear baseline with class weighting
- **Random Forest** — tree-based ensemble
- **XGBoost** — gradient boosting ensemble

Configurations are deliberately simple at this stage, to diagnose baseline capacity before any tuning is applied.

In [5]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=RANDOM_STATE
    )
}

### Baseline Evaluation

PR-AUC is used as the primary ranking metric, with ROC-AUC as a secondary signal — this dataset's severe class imbalance means ROC-AUC can appear misleadingly high even when minority-class precision is weak, while PR-AUC more directly reflects fraud detection quality.

In [6]:
baseline_results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    
    metrics = evaluate_baseline(model, X_val, y_val)
    baseline_results[name] = metrics
    
    print(f"{name}")
    print(f"  ROC-AUC: {metrics['roc_auc']:.4f}")
    print(f"  PR-AUC : {metrics['pr_auc']:.4f}\n")


Logistic Regression
  ROC-AUC: 0.9718
  PR-AUC : 0.6750

Random Forest
  ROC-AUC: 0.9597
  PR-AUC : 0.7957

XGBoost
  ROC-AUC: 0.9672
  PR-AUC : 0.8012



### Baseline Model Comparison

Results are aggregated into a comparison table, ranked by PR-AUC.

In [7]:
baseline_comparison = (
    pd.DataFrame(baseline_results)
    .T
    .sort_values("pr_auc", ascending=False)
)

baseline_comparison

,roc_auc,pr_auc
XGBoost,0.967227,0.801165
Random Forest,0.959673,0.795691
Logistic Regression,0.971811,0.675016


### Baseline Performance Interpretation

The goal at this stage is to diagnose baseline model capacity before investing in tuning, and identify which models are worth carrying forward.

**Random Forest:** the second-highest PR-AUC among the three models, with stable ensemble behavior and low sensitivity to outliers.

**XGBoost:** the highest PR-AUC of the three, indicating the strongest baseline minority-class ranking capability, alongside strong overall discrimination (ROC-AUC).

**Logistic Regression:** the highest ROC-AUC of the three models, but the lowest PR-AUC by a substantial margin. This gap is expected under severe class imbalance — ROC-AUC evaluates ranking across both classes broadly, while PR-AUC focuses specifically on minority-class precision and recall. Logistic Regression's linear decision boundary is likely less able to capture the minority class's underlying structure than the tree-based models. Precision and recall were not computed at this baseline stage; a full threshold-specific breakdown is deferred to later notebooks.

**Conclusion:** the tree-based ensemble models (Random Forest and XGBoost) show superior minority-class ranking performance and are shortlisted for hyperparameter tuning.

### My notes
# Baseline Performance Interpretation

We want to diagnose the best baseline models before investing in tuning and 
select the top performers.

##### *Random Forest:*
   
    • Second-highest PR-AUC among the three models  
    • Stable ensemble behavior, low sensitivity to outliers 
    • Stable ensemble behavior  

##### *XGBoost:*
   
    • Highest PR-AUC — best minority-class ranking capability at baseline  
    • Strong overall discrimination (ROC-AUC)

##### *Logistic Regression:*
    
    • This gap between ROC-AUC and PR-AUC is expected under severe class imbalance: ROC-AUC evaluates ranking across both classes, while PR-AUC focuses specifically on minority-class (fraud) precision and recall — Logistic Regression's linear decision boundary likely struggles more with the minority class's non-linear structure than the tree-based models do.  
    • Precision/recall were not computed at this baseline stage; a full precision-recall breakdown is deferred to threshold-specific analysis in Notebook 5.  
    • Requires threshold calibration to be viable  

#### Conclusion:

Tree-based ensemble models (Random Forest and XGBoost) demonstrate superior 
minority-class ranking performance (PR-AUC) and are shortlisted for further 
optimization.

## Persist Selected Baseline Results

Trained baseline versions of shortlisted models are saved for reuse in hyperparameter tuning and baseline vs tuned model comparison in the next notebook 

In [8]:
with open(os.path.join(RESULTS_DIR, "baseline_results.pkl"), "wb") as f:
    pickle.dump(baseline_results, f)

baseline_comparison.to_csv(
    os.path.join(RESULTS_DIR, "baseline_comparison.csv"),
    index=True
)


### Baseline Model Shortlisting


In [9]:
models_selected_for_tuning = [
    "Random Forest",
    "XGBoost"
]

models_selected_for_tuning

['Random Forest', 'XGBoost']

Based on PR-AUC (primary) and ROC-AUC (secondary), Random Forest and XGBoost advance to hyperparameter optimization; Logistic Regression is not carried forward.

### Save Shortlisted Baseline Models

Trained baseline versions of the shortlisted models are saved, enabling direct comparison against their tuned counterparts in the next notebook.

In [10]:
for model_name in models_selected_for_tuning:
    with open(
        os.path.join(MODELS_DIR, f"{model_name.replace(' ', '_').lower()}_baseline.pkl"),
        "wb"
    ) as f:
        pickle.dump(models[model_name], f)

# ---------------------------------------------END ----------------------------------

## Model selection
### Chracteristics of our dataset
* 30 features - moderate dimensionality
* PCA transformed features- Means relationships are already somewhat linear.
* Highly imbalanced data - Only some models handle this naturally (tree-based models), others need special handling.
* High recall for minority class (fraud): Missing a fraud is more dangerous than flagging a few extra legitimate transactions.
* Probabilistic output: To adjust thresholds for business needs.

## Models knowledge
* All numeric features- perfect for distance-based or linear models
* Logistic Regression gives high interpretability and strong baseline
* SVM & Neural Nets detect complex non-linear patterns
* Tree-based models capture interactions & non-linearities
* Ensemble models like XGBoost often perform best

#### Models that require scaling
* Logistic Regression - Fast, Interpretable, Works well with standardized data
* SVM - struggle on large datsets
* KNN - struggle on large datsets
* Neural Networks- MLPClassifier
* PCA-based models

#### Models that dont require scaling
* Decision Tree - Quick to train, Good for interpretability, Weak alone but useful to understand relationships
* Random Forest - Robust, Handles imbalance much better, Captures nonlinear relationships, Industry standard for tabular data
* XGBoost - Often the best performer for imbalanced tabular classification (Has scale_pos_weight for imbalance), Handles non-linearities, 
* LightGBM
* CatBoost

## 7. Baseline Model Evaluation (for quantitative summary and Ranking)

All baseline models are evaluated on the preprocessed test set using metrics suited for imbalanced datasets.

#### Why this evaluation strategy?
Fraud detection datasets are **heavily imbalanced**, therefore:
- Accuracy is misleading
- ROC-AUC can appear artificially high
- threshold-dependent metrics like precision and recall are informative

#### Evaluation metrics used:
1. **PR-AUC** (primary)
2. **F1-score** (secondary)
3. **ROC-AUC** (tertiary)

PR-AUC is prioritised because it focuses on performance for the minority (fraud) class.

Based on these metrics, the strongest baseline models are identified as:
- **Random Forest**
- **XGBoost**

* Confusion matrix or classification report are more diagnostic tools, useful when we want to analyze errors for a specific mode so we will use them in hyper parameter tuning